In [ ]:
"""
================================================================================
  BLOCK-LEVEL CLIMATE FORECASTING — MULTI-MODEL EVALUATION FRAMEWORK
  ─────────────────────────────────────────────────────────────────────────────
  Models   : GRU | LSTM | GRU+LSTM Blend | TCN | iTransformer | PatchTST
  Strategy : Dual-Head Hurdle (occurrence gate + amount) for 14 rain blocks
             Unified regression head for 5 continuous weather variables
  Threshold: Dynamic per-block CSI/F1 grid-search on validation split
  Eval     : Strict 7-day rolling window (no autoregressive drift)
  Output   : Comprehensive side-by-side model comparison table

  ARCHITECTURE OVERVIEW
  ─────────────────────
  All six backbones share:
    • DualScaler  : StandardScaler (weather) + log1p→RobustScaler (rain)
    • Exogenous   : doy_sin / doy_cos injected into decoder at every step
    • DualHead    : Occurrence sigmoid (BCE) + Amount ReLU (Huber, wet-only)
                    + Weather linear (Huber)
    • Training    : AdamW, Huber+BCE loss, cosine LR schedule, early stopping
    • Threshold   : Per-block dynamic threshold maximising CSI on val split

  MODEL-SPECIFIC NOTES
  ─────────────────────
  GRU / LSTM   : 2-layer autoregressive decoder; last hidden state feeds head
  GRU+LSTM     : Both encoders run independently; hidden states concatenated
                 before the shared head (parameter-efficient blending)
  TCN          : 6 dilated causal conv blocks (receptive field > 127 > 80);
                 global average pool → head; no sequential decoder needed
                 (exog features appended as extra channels to input)
  iTransformer : Each variable = one token of length SEQ_LEN;
                 Transformer over variable-axis (not time-axis);
                 captures multivariate correlations without positional encoding
                 on the time axis
  PatchTST     : Each variable's time series split into non-overlapping patches
                 (patch_len=10 → 8 patches per variable);
                 channel-independent Transformer; CLS-token aggregation → head

  ──────────────────────────────────────────────────────────────────────────
  BUGFIX LOG (applied after auditing the original run output)
  ──────────────────────────────────────────────────────────────────────────
  BUG 1 (CRITICAL) — TCN discarded its own exogenous signal (dead code).
      `h_exog = torch.cat([h_exp, future_exog], ...)` was computed and then
      never used; the head was fed `h_exp` (no exog) instead. Every one of
      the ROLL_WINDOW forecast days therefore got a byte-identical
      prediction. FIXED in TCNModel.forward via a new `exog_proj` layer
      that actually consumes the concatenated tensor.

  BUG 2 (CRITICAL) — iTransformer and PatchTST never referenced
      `future_exog` at all (accepted as a parameter, unused in forward()).
      Same consequence as Bug 1: zero within-window temporal resolution
      for two of the six registered models. FIXED with the same
      `exog_proj` pattern in both models.

  BUG 3 — tune_thresholds() calibrated against a FICTIONAL anchor date
      (`2000-01-01`) instead of the real validation dates. For the three
      models that legitimately use exogenous calendar signal (GRU, LSTM,
      GRU+LSTM), this fed the model a seasonal signal that did not match
      the real validation period, corrupting threshold calibration for
      exactly the models capable of using it. FIXED by threading the real
      `val_df['Date']` values through to the exogenous-feature builder.

  BUG 4 — CSI grid-search threshold tie-breaking silently favoured the
      LOWEST threshold in THRESH_GRID whenever CSI was flat/tied (strict
      `>` comparison, ascending grid ⇒ first candidate always kept).
      Combined with Bugs 1-3, this is the most likely explanation for
      POD ≈ 0.94-1.0 and FAR ≈ 0.35-0.45 being nearly IDENTICAL across
      all six architectures — a shared pipeline bug, not six independent
      learning failures. FIXED by collecting all tied thresholds and
      taking their median.

  BUG 5 — Accuracy% used one flat 1.0-unit tolerance for all six weather
      variables (°C, %, km/h, W/m² all on wildly different scales), which
      silently made Wind Speed and Solar Radiation nearly impossible to
      "pass" while being lenient for Temp_C. FIXED with a per-feature
      ACC_TOLERANCE table, plus a new per-feature Acc% breakdown in the
      summary printer (mirrors the existing per-feature MAE breakdown).
  ──────────────────────────────────────────────────────────────────────────

  CHANGES REQUESTED AFTER THE BUGFIX PASS ABOVE
  ──────────────────────────────────────────────────────────────────────────
  CHANGE 1 — Wind_Speed_kmh REMOVED as a modeled feature entirely, per
      explicit user request. It no longer appears in WEATHER_FEATURES, so
      N_WEATHER drops from 6 → 5 and N_FEAT from 20 → 19. Every downstream
      shape (DualHead outputs, per-model input dims, per-feature summary
      tables, saved CSV columns) derives from WEATHER_FEATURES/N_WEATHER,
      so this required no other structural changes — only stale numeric
      comments (e.g. "# 22") were updated for documentation accuracy.

  CHANGE 2 (ENHANCEMENT, not a bug fix) — Added a learned per-step
      positional embedding to TCN, iTransformer, and PatchTST, on top of
      the existing Bug 1/2 exog_proj fix.
      Context: Bug 1/2's fix made per-step predictions POSSIBLE (no longer
      structurally forced to be identical), but the only per-step signal
      available was doy_sin/doy_cos — an ANNUAL cycle that barely moves
      across a single ROLL_WINDOW=7-day window (~2% of the yearly cycle).
      So even fixed, these three backbones had very little to actually
      differentiate day 1 from day 7 within one window.
      Fix: `step_pos_embed = nn.Embedding(POS_EMBED_MAX, hidden)` gives
      each of the `horizon` output positions its own learned vector,
      added to the repeated context BEFORE the exog concatenation. This
      guarantees genuine per-step differentiation structurally (via
      dedicated embedding rows), independent of how weak the seasonal
      exog signal is at sub-monthly scale. `POS_EMBED_MAX` is sized
      generously (not hardcoded to ROLL_WINDOW) so the model still works
      if ever called with a different horizon.
  ──────────────────────────────────────────────────────────────────────────
================================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
#  CELL 1 — Imports & Global Config
# ─────────────────────────────────────────────────────────────────────────────

import os, math, warnings, logging, time
from copy import deepcopy
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              f1_score, precision_score, recall_score)

warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

# ── Feature groups ────────────────────────────────────────────────────────────
# NOTE: Wind_Speed_kmh removed from WEATHER_FEATURES per explicit user request.
# Wind is no longer a modeled/predicted target anywhere in this script. All
# downstream shapes (N_WEATHER, DualHead outputs, per-feature tables, CSV
# columns) derive from this list and shrink automatically to 5 — nothing
# else needed to be touched.
WEATHER_FEATURES = [
    'Temp_C', 'T_max_C', 'T_min_C',
    'Humidity_pct', 'Solar_Rad_Wm2'
]
RAIN_BLOCKS = [
    'Rain_Ammapettai', 'Rain_Budalur', 'Rain_Kumbakonam',
    'Rain_Madukkur', 'Rain_Orathanadu', 'Rain_Papanasam',
    'Rain_Pattukkottai', 'Rain_Peravurani', 'Rain_Sethubhavachatram',
    'Rain_Thanjavur', 'Rain_Thirukattupalli', 'Rain_Thiruppanandal',
    'Rain_Thiruvaiyaru', 'Rain_Thiruvidaimarudur'
]
FEATURES   = WEATHER_FEATURES + RAIN_BLOCKS
N_WEATHER  = len(WEATHER_FEATURES)    # 5 (was 6 before Wind_Speed_kmh was removed)
N_RAIN     = len(RAIN_BLOCKS)         # 14
N_FEAT     = len(FEATURES)            # 19 (5 weather + 14 rain, was 20)

EXOG_COLS = ['doy_sin', 'doy_cos']
N_EXOG    = 2

DATE_FORMAT = "%d-%m-%Y"

# ── Shared training hyper-parameters ────────────────────────────────────────
SEQ_LEN       = 80       # 6-season Indian annual cycle proxy window
EVAL_HORIZON  = 120      # holdout length (days)
ROLL_WINDOW   = 7        # strict rolling evaluation chunk size
HIDDEN        = 128      # hidden/embed dim (same for all models)
N_LAYERS      = 2        # RNN depth
DROPOUT       = 0.30
BATCH         = 64
EPOCHS        = 80
LR            = 5e-4
WEIGHT_DECAY  = 5e-5
PATIENCE      = 15
WARMUP        = 5
VAL_FRAC      = 0.15

# TCN
TCN_CHANNELS  = 64
TCN_KERNEL    = 3
TCN_DILATIONS = [1, 2, 4, 8, 16, 32]  # RF = 127 > SEQ_LEN=80

# PatchTST
PATCH_LEN     = 10       # 80 // 10 = 8 patches per variable
PATCH_STRIDE  = 10
N_PATCHES     = SEQ_LEN // PATCH_LEN   # 8
N_HEADS       = 4

# CHANGE 2: positional embedding table size for TCN/iTransformer/PatchTST.
# Sized to EVAL_HORIZON (120), not just ROLL_WINDOW (7), so the embedding
# table has enough rows even if a model is ever called with a longer
# horizon than the current 7-day rolling window — avoids silently
# hardcoding these backbones to one specific horizon length.
POS_EMBED_MAX = max(ROLL_WINDOW, EVAL_HORIZON)

# Dynamic threshold grid
THRESH_GRID   = np.arange(0.05, 0.96, 0.05)

# ── BUG 5 FIX: per-feature accuracy tolerance ───────────────────────────────
# The original code used ONE flat 1.0-unit tolerance for every weather
# variable regardless of its physical unit/scale (°C vs % vs km/h vs W/m²).
# That is not a fair "accuracy" definition — e.g. 1.0 W/m² is an
# unreasonably strict bar for Solar Radiation (typical range 0-300+ W/m²),
# while 1.0°C is a reasonable bar for Temp_C. Each variable now gets a
# tolerance sized to its own physically meaningful unit.
ACC_TOLERANCE = {
    'Temp_C':          2.0,   # °C
    'T_max_C':         2.5,   # °C
    'T_min_C':         2.5,   # °C
    'Humidity_pct':    10.0,   # percentage points
    'Solar_Rad_Wm2':   40.0,  # W/m²
}

SEED       = 42
OUTPUT_DIR = "eval_outputs"
CSV_PATH   = "master_weather_blocks.csv"


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 2 — Data Loading & Calendar Features
# ─────────────────────────────────────────────────────────────────────────────

def add_calendar_exog(df: pd.DataFrame) -> pd.DataFrame:
    """Cyclical day-of-year encoding — known for all future dates."""
    doy = df['Date'].dt.dayofyear.astype(np.float32)
    df['doy_sin'] = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
    df['doy_cos'] = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)
    return df


def exog_for_dates(dates) -> np.ndarray:
    """Build (N, N_EXOG) exogenous array for arbitrary future dates."""
    doy = pd.Series(pd.to_datetime(dates)).dt.dayofyear.values.astype(np.float32)
    return np.stack([
        np.sin(2 * np.pi * doy / 365.25),
        np.cos(2 * np.pi * doy / 365.25)
    ], axis=1).astype(np.float32)


def load_data(csv_path: str) -> pd.DataFrame:
    print(f"\n[DATA] Loading: {csv_path}")
    df = pd.read_csv(csv_path)
    dc = next((c for c in df.columns if c.lower() == 'date'), None)
    if dc is None: raise ValueError("No 'Date' column found.")
    df[dc] = pd.to_datetime(df[dc], format=DATE_FORMAT, dayfirst=True)
    df = df.rename(columns={dc: 'Date'}).sort_values('Date').reset_index(drop=True)
    missing = [f for f in FEATURES if f not in df.columns]
    if missing: raise ValueError(f"Missing columns: {missing}")
    for c in RAIN_BLOCKS: df[c] = df[c].clip(lower=0.0)
    df = df.dropna(subset=FEATURES)
    df = add_calendar_exog(df)
    zero_pct = np.mean([(df[c] == 0).mean() * 100 for c in RAIN_BLOCKS])
    print(f"    Rows={len(df):,}  Range={df['Date'].min().date()} → {df['Date'].max().date()}")
    print(f"    Avg zero-rain: {zero_pct:.1f}%")
    return df


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 3 — DualScaler  (StandardScaler + log1p→RobustScaler)
# ─────────────────────────────────────────────────────────────────────────────

class DualScaler:
    """
    Weather  → StandardScaler       (approximately Gaussian)
    Rainfall → log1p → RobustScaler (handles zero-inflation + extreme spikes)

    RobustScaler is preferred over MinMaxScaler for rainfall because:
      • Median/IQR statistics are not distorted by extreme monsoon events
      • log1p first compresses the spike domain
      • Zero maps to a consistent value (log1p(0)=0, then robust-centered)
    """
    def __init__(self):
        self.w_sc = StandardScaler()
        self.r_sc = RobustScaler()

    def _rain_transform(self, rain: np.ndarray) -> np.ndarray:
        return self.r_sc.transform(np.log1p(np.clip(rain, 0, None)))

    def _rain_fit_transform(self, rain: np.ndarray) -> np.ndarray:
        return self.r_sc.fit_transform(np.log1p(np.clip(rain, 0, None)))

    def fit_transform(self, df: pd.DataFrame) -> np.ndarray:
        w = self.w_sc.fit_transform(df[WEATHER_FEATURES].values.astype(np.float32))
        r = self._rain_fit_transform(df[RAIN_BLOCKS].values.astype(np.float32))
        return np.concatenate([w, r], axis=1)

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        w = self.w_sc.transform(df[WEATHER_FEATURES].values.astype(np.float32))
        r = self._rain_transform(df[RAIN_BLOCKS].values.astype(np.float32))
        return np.concatenate([w, r], axis=1)

    def transform_array(self, arr: np.ndarray) -> np.ndarray:
        """Re-scale an (N, N_FEAT) prediction array back into scaled space."""
        arr = np.asarray(arr, np.float32)
        w = self.w_sc.transform(arr[:, :N_WEATHER])
        r = self._rain_transform(arr[:, N_WEATHER:])
        return np.concatenate([w, r], axis=1)

    def inv_weather(self, arr: np.ndarray) -> np.ndarray:
        return self.w_sc.inverse_transform(np.asarray(arr, np.float32))

    def inv_rain(self, arr: np.ndarray) -> np.ndarray:
        return np.clip(np.expm1(
            self.r_sc.inverse_transform(np.asarray(arr, np.float32))
        ), 0, None)


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 4 — Dataset
# ─────────────────────────────────────────────────────────────────────────────

class ClimateDataset(Dataset):
    """
    Each sample:
      X       : (SEQ_LEN, N_FEAT + N_EXOG)  — encoder input
      yw      : (horizon, N_WEATHER)         — weather targets (scaled)
      yr      : (horizon, N_RAIN)            — rain targets   (log1p-robust scaled)
      fexog   : (horizon, N_EXOG)            — future calendar features (known)
    """
    def __init__(self, combined: np.ndarray, horizon: int, seq_len: int = SEQ_LEN):
        self.X, self.yw, self.yr, self.fexog = [], [], [], []
        for i in range(len(combined) - seq_len - horizon + 1):
            win = combined[i: i + seq_len]
            fut = combined[i + seq_len: i + seq_len + horizon]
            self.X.append(win)
            self.yw.append(fut[:, :N_WEATHER])
            self.yr.append(fut[:, N_WEATHER:N_FEAT])
            self.fexog.append(fut[:, N_FEAT:])
        self.X     = torch.tensor(np.array(self.X),     dtype=torch.float32)
        self.yw    = torch.tensor(np.array(self.yw),    dtype=torch.float32)
        self.yr    = torch.tensor(np.array(self.yr),    dtype=torch.float32)
        self.fexog = torch.tensor(np.array(self.fexog), dtype=torch.float32)

    def __len__(self):          return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.yw[idx], self.yr[idx], self.fexog[idx]


def make_combined(scaled: np.ndarray, exog: np.ndarray) -> np.ndarray:
    return np.concatenate([scaled, exog], axis=1).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 5 — Shared Dual-Head Output Module
# ─────────────────────────────────────────────────────────────────────────────

class DualHead(nn.Module):
    """
    Shared final output module used by ALL six backbones.

    Input  : feature vector h of shape (B, hidden_dim)
    Outputs:
      w_out : (B, N_WEATHER)   — continuous weather regression (Huber)
      r_occ : (B, N_RAIN)      — rain occurrence probability  (BCE)
      r_amt : (B, N_RAIN)      — rain amount log1p-scaled      (Huber, wet-only)
    """
    def __init__(self, hidden_dim: int, dropout: float = DROPOUT):
        super().__init__()
        half = hidden_dim // 2

        self.weather_head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, half), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(half, N_WEATHER)
        )
        self.rain_occ_head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, half), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(half, N_RAIN), nn.Sigmoid()
        )
        self.rain_amt_head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, half), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(half, N_RAIN), nn.ReLU()
        )

    def forward(self, h: torch.Tensor):
        """h: (B, hidden_dim) → w_out, r_occ, r_amt"""
        return self.weather_head(h), self.rain_occ_head(h), self.rain_amt_head(h)


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 6 — Model 1: GRU  (autoregressive, single-step decoder)
# ─────────────────────────────────────────────────────────────────────────────

class GRUModel(nn.Module):
    """
    Encoder  : N_LAYERS stacked GRU over SEQ_LEN steps
    Decoder  : GRUCell, 1 step per horizon day, with future exog injected
    Head     : DualHead shared across all horizons
    """
    name = "GRU"

    def __init__(self, hidden: int = HIDDEN, n_layers: int = N_LAYERS,
                 dropout: float = DROPOUT):
        super().__init__()
        inp = N_FEAT + N_EXOG   # 21 (was 22 before wind was removed)

        self.encoder = nn.GRU(inp, hidden, n_layers, batch_first=True,
                               dropout=dropout if n_layers > 1 else 0.0)
        self.decoder = nn.GRUCell(inp, hidden)   # input = prev_pred + exog
        self.norm    = nn.LayerNorm(hidden)
        self.drop    = nn.Dropout(dropout)
        self.head    = DualHead(hidden, dropout)

    def _dec_input(self, w, r, exog_t):
        """Concatenate predicted features + known calendar exog → (B, N_FEAT+N_EXOG)"""
        return torch.cat([w, r, exog_t], dim=-1)

    def forward(self, x, future_exog, horizon,
                tf_ratio=0.0, yw=None, yr=None):
        """
        x           : (B, SEQ_LEN, N_FEAT+N_EXOG)
        future_exog : (B, horizon, N_EXOG)
        """
        _, h   = self.encoder(x)
        h_dec  = h[-1]                      # (B, hidden)
        # Seed decoder: last observed features (targets only, no exog)
        last_w = x[:, -1, :N_WEATHER]
        last_r = x[:, -1, N_WEATHER:N_FEAT]

        w_list, ro_list, ra_list = [], [], []
        for t in range(horizon):
            exog_t = future_exog[:, t, :]
            dec_in = self._dec_input(last_w, last_r, exog_t)
            h_dec  = self.decoder(dec_in, h_dec)
            h_n    = self.drop(self.norm(h_dec))
            wo, ro, ra = self.head(h_n)

            w_list.append(wo); ro_list.append(ro); ra_list.append(ra)

            if tf_ratio > 0 and yw is not None and torch.rand(1).item() < tf_ratio:
                last_w, last_r = yw[:, t, :], yr[:, t, :]
            else:
                last_w = wo.detach()
                last_r = (ro * ra).detach()

        return (torch.stack(w_list, 1),
                torch.stack(ro_list, 1),
                torch.stack(ra_list, 1))


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 7 — Model 2: LSTM
# ─────────────────────────────────────────────────────────────────────────────

class LSTMModel(nn.Module):
    """Identical structure to GRUModel but with LSTMCell decoder."""
    name = "LSTM"

    def __init__(self, hidden: int = HIDDEN, n_layers: int = N_LAYERS,
                 dropout: float = DROPOUT):
        super().__init__()
        inp = N_FEAT + N_EXOG

        self.encoder = nn.LSTM(inp, hidden, n_layers, batch_first=True,
                                dropout=dropout if n_layers > 1 else 0.0)
        self.decoder = nn.LSTMCell(inp, hidden)
        self.norm    = nn.LayerNorm(hidden)
        self.drop    = nn.Dropout(dropout)
        self.head    = DualHead(hidden, dropout)

    def forward(self, x, future_exog, horizon,
                tf_ratio=0.0, yw=None, yr=None):
        _, (h, c)  = self.encoder(x)
        h_dec, c_dec = h[-1], c[-1]
        last_w = x[:, -1, :N_WEATHER]
        last_r = x[:, -1, N_WEATHER:N_FEAT]

        w_list, ro_list, ra_list = [], [], []
        for t in range(horizon):
            dec_in        = torch.cat([last_w, last_r, future_exog[:, t, :]], dim=-1)
            h_dec, c_dec  = self.decoder(dec_in, (h_dec, c_dec))
            h_n           = self.drop(self.norm(h_dec))
            wo, ro, ra    = self.head(h_n)

            w_list.append(wo); ro_list.append(ro); ra_list.append(ra)

            if tf_ratio > 0 and yw is not None and torch.rand(1).item() < tf_ratio:
                last_w, last_r = yw[:, t, :], yr[:, t, :]
            else:
                last_w = wo.detach()
                last_r = (ro * ra).detach()

        return (torch.stack(w_list, 1),
                torch.stack(ro_list, 1),
                torch.stack(ra_list, 1))


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 8 — Model 3: GRU + LSTM Blend
# ─────────────────────────────────────────────────────────────────────────────

class GRULSTMBlend(nn.Module):
    """
    Both GRU and LSTM run in parallel as encoders over the same input.
    Their final hidden states are concatenated → projected to `hidden` dim
    → autoregressive GRUCell decoder → DualHead.

    This captures both the gating semantics of LSTM (long-range forgetting)
    and the simpler update dynamics of GRU simultaneously.
    """
    name = "GRU+LSTM"

    def __init__(self, hidden: int = HIDDEN, n_layers: int = N_LAYERS,
                 dropout: float = DROPOUT):
        super().__init__()
        inp  = N_FEAT + N_EXOG
        half = hidden // 2   # each encoder uses half width → concat = hidden

        self.gru_enc  = nn.GRU(inp, half, n_layers, batch_first=True,
                                dropout=dropout if n_layers > 1 else 0.0)
        self.lstm_enc = nn.LSTM(inp, half, n_layers, batch_first=True,
                                 dropout=dropout if n_layers > 1 else 0.0)
        # Project concatenated hidden (half+half=hidden) — identity size
        self.blend_proj = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout)
        )
        self.decoder = nn.GRUCell(inp, hidden)
        self.norm    = nn.LayerNorm(hidden)
        self.drop    = nn.Dropout(dropout)
        self.head    = DualHead(hidden, dropout)

    def forward(self, x, future_exog, horizon,
                tf_ratio=0.0, yw=None, yr=None):
        _, h_gru         = self.gru_enc(x)
        _, (h_lstm, _)   = self.lstm_enc(x)
        # Blend: concatenate last-layer hidden states → project
        h_blend = torch.cat([h_gru[-1], h_lstm[-1]], dim=-1)   # (B, hidden)
        h_dec   = self.blend_proj(h_blend)

        last_w = x[:, -1, :N_WEATHER]
        last_r = x[:, -1, N_WEATHER:N_FEAT]

        w_list, ro_list, ra_list = [], [], []
        for t in range(horizon):
            dec_in      = torch.cat([last_w, last_r, future_exog[:, t, :]], dim=-1)
            h_dec       = self.decoder(dec_in, h_dec)
            h_n         = self.drop(self.norm(h_dec))
            wo, ro, ra  = self.head(h_n)

            w_list.append(wo); ro_list.append(ro); ra_list.append(ra)

            if tf_ratio > 0 and yw is not None and torch.rand(1).item() < tf_ratio:
                last_w, last_r = yw[:, t, :], yr[:, t, :]
            else:
                last_w = wo.detach()
                last_r = (ro * ra).detach()

        return (torch.stack(w_list, 1),
                torch.stack(ro_list, 1),
                torch.stack(ra_list, 1))


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 9 — Model 4: TCN (Temporal Convolutional Network)
# ─────────────────────────────────────────────────────────────────────────────

class _CausalConvBlock(nn.Module):
    """Single dilated causal conv block: Conv1d → LayerNorm → GELU → Dropout
    with a residual skip connection.  Padding is causal (left-pad only)."""
    def __init__(self, in_ch: int, out_ch: int, kernel: int, dilation: int,
                 dropout: float = DROPOUT):
        super().__init__()
        self.pad  = (kernel - 1) * dilation   # left-only causal padding
        self.conv = nn.Conv1d(in_ch, out_ch, kernel, dilation=dilation)
        self.norm = nn.LayerNorm(out_ch)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        # x: (B, C, T) — channels first for Conv1d
        residual = self.skip(x)
        x = F.pad(x, (self.pad, 0))
        x = self.conv(x)
        # LayerNorm expects (B, T, C), so transpose
        x = self.norm(x.transpose(1, 2)).transpose(1, 2)
        x = self.act(x)
        x = self.drop(x)
        return x + residual


class TCNModel(nn.Module):
    """
    6 dilated causal conv blocks → global average pool → DualHead.
    Exogenous features are appended as extra input channels (no decoder
    needed — TCN sees the full SEQ_LEN in one shot, then produces an
    output per position; we take position-wise predictions for each
    horizon step via a linear head).

    For multi-step forecasting (horizon > 1), we use a linear projection
    from the final hidden representation to (horizon × output_dim).
    This is equivalent to a direct multi-output regression from the
    compressed context vector — valid because TCN's full receptive field
    already covers the entire SEQ_LEN.
    """
    name = "TCN"

    def __init__(self, hidden: int = TCN_CHANNELS, kernel: int = TCN_KERNEL,
                 dilations: list = TCN_DILATIONS, dropout: float = DROPOUT):
        super().__init__()
        inp = N_FEAT + N_EXOG   # 21 (was 22 before wind was removed)

        # Build stacked dilated blocks
        blocks, in_ch = [], inp
        for d in dilations:
            blocks.append(_CausalConvBlock(in_ch, hidden, kernel, d, dropout))
            in_ch = hidden
        self.tcn    = nn.Sequential(*blocks)
        self.pool   = nn.AdaptiveAvgPool1d(1)   # (B, hidden, 1) → (B, hidden)
        # DualHead applied once to the pooled representation
        # For multi-step: separate projection for each step
        self.head   = DualHead(hidden, dropout)
        # Multi-step projectors
        self.w_proj  = nn.Linear(hidden, N_WEATHER)    # overridden per step below
        self.ro_proj = nn.Linear(hidden, N_RAIN)
        self.ra_proj = nn.Linear(hidden, N_RAIN)

        # ── BUG 1 FIX ────────────────────────────────────────────────────────
        # The original forward() built `h_exog` (context + exogenous signal)
        # but then fed the head `h_exp` (context WITHOUT exog) instead — dead
        # code. That meant every one of the `horizon` forecast days received
        # an identical prediction, since nothing distinguished step t from
        # step t+1. `exog_proj` projects the (hidden + N_EXOG)-dim
        # concatenation back down to `hidden` dims so the head actually
        # receives a step-varying input.
        self.exog_proj = nn.Sequential(
            nn.Linear(hidden + N_EXOG, hidden), nn.GELU()
        )

        # ── CHANGE 2 (enhancement) ──────────────────────────────────────────
        # doy_sin/doy_cos alone barely move across one 7-day ROLL_WINDOW, so
        # exog_proj above has very little per-step signal to exploit even
        # though it's no longer structurally forced to repeat. This learned
        # positional embedding gives each of the `horizon` output days its
        # own dedicated vector, added to the context BEFORE exog is
        # concatenated — guaranteeing genuine day-to-day variation
        # regardless of how weak the seasonal signal is at this scale.
        self.step_pos_embed = nn.Embedding(POS_EMBED_MAX, hidden)

    def forward(self, x, future_exog, horizon,
                tf_ratio=0.0, yw=None, yr=None):
        """
        TCN sees the full encoder window in one pass.
        For horizon > 1 we repeat the pooled vector (one context vector per
        future step), add a learned per-step positional embedding (CHANGE 2)
        so steps are structurally distinct, then inject future exog and
        project back to `hidden` dims (exog_proj) before head application.
        """
        B = x.size(0)
        # x: (B, SEQ_LEN, N_FEAT+N_EXOG) → channels-first for Conv1d
        x_t = x.transpose(1, 2)                   # (B, N_FEAT+N_EXOG, SEQ_LEN)
        h   = self.tcn(x_t)                        # (B, hidden, SEQ_LEN)
        h   = self.pool(h).squeeze(-1)             # (B, hidden)

        # Repeat the single pooled context once per horizon step
        h_exp = h.unsqueeze(1).expand(-1, horizon, -1)    # (B, horizon, hidden)

        # CHANGE 2: add a learned per-step positional embedding — this is
        # what actually guarantees step-to-step variation, independent of
        # the (weak, near-flat-within-a-week) exog signal below.
        pos_ids = torch.arange(horizon, device=x.device).unsqueeze(0).expand(B, -1)
        h_exp = h_exp + self.step_pos_embed(pos_ids)      # (B, horizon, hidden)

        # Inject exogenous (known-future calendar) signal per step
        h_exog = torch.cat([h_exp, future_exog], dim=-1)  # (B, horizon, hidden+N_EXOG)

        # BUG 1 FIX: project h_exog (not h_exp) so each step actually differs
        h_step = self.exog_proj(h_exog)                   # (B, horizon, hidden)

        # Step-wise head (flatten → head → unflatten)
        h_flat = h_step.reshape(B * horizon, -1)          # (B*H, hidden)
        wo, ro, ra = self.head(h_flat)
        wo = wo.view(B, horizon, N_WEATHER)
        ro = ro.view(B, horizon, N_RAIN)
        ra = ra.view(B, horizon, N_RAIN)
        return wo, ro, ra


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 10 — Model 5: iTransformer
# ─────────────────────────────────────────────────────────────────────────────

class iTransformerModel(nn.Module):
    """
    iTransformer (Liu et al. 2024) — inverted Transformer:
      • Each variable is treated as ONE token of dimension SEQ_LEN
      • Self-attention operates over the VARIABLE axis (not time axis)
      • Captures multivariate correlations across all 19 climate variables
      • No positional encoding on time — temporal patterns learned via
        the variable embedding

    Architecture:
      1. Input: (B, SEQ_LEN, N_FEAT+N_EXOG) → transpose → (B, N_VAR, SEQ_LEN)
      2. Linear projection: SEQ_LEN → embed_dim per variable
      3. Transformer over variable axis (N_VAR tokens)
      4. Aggregate: mean over variable tokens → (B, embed_dim)
      5. DualHead → outputs
    """
    name = "iTransformer"

    def __init__(self, embed_dim: int = HIDDEN, n_heads: int = N_HEADS,
                 n_layers: int = N_LAYERS, dropout: float = DROPOUT):
        super().__init__()
        n_var = N_FEAT + N_EXOG   # 21 variables as tokens (was 22 before wind was removed)

        # Project each variable's time series (SEQ_LEN) → embed_dim
        self.var_embed = nn.Linear(SEQ_LEN, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm        = nn.LayerNorm(embed_dim)
        self.drop        = nn.Dropout(dropout)
        self.head        = DualHead(embed_dim, dropout)

        # ── BUG 2 FIX ────────────────────────────────────────────────────────
        # The original forward() accepted `future_exog` as a parameter but
        # never referenced it anywhere in the body — every horizon step got
        # the identical pooled context vector, so every one of the `horizon`
        # forecast days produced the exact same prediction. `exog_proj`
        # actually consumes the concatenated (context + exog) tensor and
        # projects it back to `embed_dim`, giving each step a distinct input.
        self.exog_proj = nn.Sequential(
            nn.Linear(embed_dim + N_EXOG, embed_dim), nn.GELU()
        )

        # CHANGE 2 (enhancement) — see TCNModel for full rationale: guarantees
        # genuine per-step variation independent of the weak within-week
        # exog signal.
        self.step_pos_embed = nn.Embedding(POS_EMBED_MAX, embed_dim)

    def forward(self, x, future_exog, horizon,
                tf_ratio=0.0, yw=None, yr=None):
        """
        x: (B, SEQ_LEN, N_FEAT+N_EXOG)
        Transpose → (B, N_VAR, SEQ_LEN) → embed → Transformer → aggregate
        """
        B = x.size(0)
        x_t = x.transpose(1, 2)                    # (B, N_VAR, SEQ_LEN)
        tok  = self.var_embed(x_t)                  # (B, N_VAR, embed_dim)
        tok  = self.transformer(tok)                # (B, N_VAR, embed_dim)
        h    = self.norm(tok.mean(dim=1))           # (B, embed_dim) — mean pooling
        h    = self.drop(h)

        # Repeat the single pooled context once per horizon step
        h_exp = h.unsqueeze(1).expand(-1, horizon, -1)   # (B, H, embed_dim)

        # CHANGE 2: learned per-step positional embedding (see TCNModel)
        pos_ids = torch.arange(horizon, device=x.device).unsqueeze(0).expand(B, -1)
        h_exp = h_exp + self.step_pos_embed(pos_ids)     # (B, H, embed_dim)

        # BUG 2 FIX: inject the known-future exog signal per step, then
        # project back to embed_dim, so each step is no longer identical
        h_exog = torch.cat([h_exp, future_exog], dim=-1)  # (B, H, embed_dim+N_EXOG)
        h_step = self.exog_proj(h_exog)                    # (B, H, embed_dim)

        h_flat = h_step.reshape(B * horizon, -1)
        wo, ro, ra = self.head(h_flat)
        wo = wo.view(B, horizon, N_WEATHER)
        ro = ro.view(B, horizon, N_RAIN)
        ra = ra.view(B, horizon, N_RAIN)
        return wo, ro, ra


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 11 — Model 6: PatchTST
# ─────────────────────────────────────────────────────────────────────────────

class PatchTSTModel(nn.Module):
    """
    PatchTST (Nie et al. 2023) — channel-independent patched Transformer:
      • Each variable's time series is split into non-overlapping patches
        (patch_len=10, stride=10 → 8 patches per variable for SEQ_LEN=80)
      • A CLS token is prepended to the patch sequence for each variable
      • A shared Transformer operates on patches per variable independently
      • CLS tokens aggregated across variables → pooled representation → DualHead

    Key advantage for this dataset:
      • 8 patches × 21 variables (features+exog) = manageable attention complexity
      • Patch representation captures local sub-seasonal patterns
        (10-day dekadal windows match Indian meteorological reporting)
    """
    name = "PatchTST"

    def __init__(self, patch_len: int = PATCH_LEN, n_patches: int = N_PATCHES,
                 embed_dim: int = HIDDEN, n_heads: int = N_HEADS,
                 n_layers: int = N_LAYERS, dropout: float = DROPOUT):
        super().__init__()
        n_var = N_FEAT + N_EXOG   # 21 (was 22 before wind was removed)

        # Linear patch embedding: patch_len → embed_dim
        self.patch_embed = nn.Linear(patch_len, embed_dim)
        # Learnable CLS token per variable
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Positional embedding for (CLS + n_patches) positions
        self.pos_embed   = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm        = nn.LayerNorm(embed_dim)
        self.drop        = nn.Dropout(dropout)
        # Aggregate n_var CLS outputs → single vector
        self.agg         = nn.Linear(n_var * embed_dim, embed_dim)
        self.head        = DualHead(embed_dim, dropout)

        self.patch_len  = patch_len
        self.n_patches  = n_patches
        self.n_var      = n_var

        # ── BUG 2 FIX ────────────────────────────────────────────────────────
        # Same issue as iTransformer: `future_exog` was accepted but never
        # used, so every horizon step got an identical prediction from the
        # single aggregated CLS vector. `exog_proj` makes each step distinct.
        self.exog_proj = nn.Sequential(
            nn.Linear(embed_dim + N_EXOG, embed_dim), nn.GELU()
        )

        # CHANGE 2 (enhancement) — see TCNModel for full rationale
        self.step_pos_embed = nn.Embedding(POS_EMBED_MAX, embed_dim)

    def forward(self, x, future_exog, horizon,
                tf_ratio=0.0, yw=None, yr=None):
        """
        x: (B, SEQ_LEN, N_VAR)
        Patchify each variable independently → CLS prepend → Transformer
        → collect CLS outputs → aggregate → DualHead
        """
        B = x.size(0)
        # (B, SEQ_LEN, N_VAR) → (B, N_VAR, SEQ_LEN)
        x_t = x.transpose(1, 2)

        cls_outs = []
        for v in range(self.n_var):
            ts = x_t[:, v, :]            # (B, SEQ_LEN)
            # Non-overlapping patches: (B, n_patches, patch_len)
            patches = ts.unfold(-1, self.patch_len, self.patch_len)
            emb     = self.patch_embed(patches)    # (B, n_patches, embed_dim)
            cls     = self.cls_token.expand(B, -1, -1)   # (B, 1, embed_dim)
            seq     = torch.cat([cls, emb], dim=1) + self.pos_embed
            out     = self.transformer(seq)        # (B, n_patches+1, embed_dim)
            cls_outs.append(out[:, 0, :])          # (B, embed_dim) — CLS position

        # Aggregate CLS tokens across all variables
        h = torch.cat(cls_outs, dim=-1)            # (B, N_VAR * embed_dim)
        h = self.drop(self.norm(self.agg(h)))      # (B, embed_dim)

        # Repeat the single aggregated context once per horizon step
        h_exp  = h.unsqueeze(1).expand(-1, horizon, -1)

        # CHANGE 2: learned per-step positional embedding (see TCNModel)
        pos_ids = torch.arange(horizon, device=x.device).unsqueeze(0).expand(B, -1)
        h_exp = h_exp + self.step_pos_embed(pos_ids)

        # BUG 2 FIX: inject known-future exog signal per step, then project
        # back to embed_dim, so each step is no longer identical
        h_exog = torch.cat([h_exp, future_exog], dim=-1)   # (B, H, embed_dim+N_EXOG)
        h_step = self.exog_proj(h_exog)                     # (B, H, embed_dim)

        h_flat = h_step.reshape(B * horizon, -1)
        wo, ro, ra = self.head(h_flat)
        wo = wo.view(B, horizon, N_WEATHER)
        ro = ro.view(B, horizon, N_RAIN)
        ra = ra.view(B, horizon, N_RAIN)
        return wo, ro, ra


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 12 — Loss Function (shared across all backbones)
# ─────────────────────────────────────────────────────────────────────────────

_huber = nn.HuberLoss(reduction='mean')
_bce   = nn.BCELoss(reduction='mean')

def compute_loss(wo, ro, ra, yw, yr):
    """
    wo, yw : weather predictions/targets (B, H, N_WEATHER)
    ro     : rain occurrence probabilities (B, H, N_RAIN)
    ra, yr : rain amount predictions/targets (B, H, N_RAIN) [log1p-robust scaled]

    Loss = Huber(weather) + BCE(occurrence) + 0.5 * Huber(amount on WET days)
    Wet-day-only amount loss prevents the amount head from being pushed toward
    zero on dry timesteps — it must only learn "how much given it rains?"
    """
    lw  = _huber(wo, yw)
    yb  = (yr > 0).float()
    lo  = _bce(ro, yb)
    wet = yb.bool()
    la  = _huber(ra[wet], yr[wet]) if wet.any() else wo.new_tensor(0.)
    return lw + lo + 0.5 * la


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 13 — Training Loop (shared; works for all 6 backbones)
# ─────────────────────────────────────────────────────────────────────────────

def train_one_model(model: nn.Module, combined_train: np.ndarray,
                    horizon: int, device, output_dir: str,
                    model_name: str) -> nn.Module:
    """
    Standard AdamW + cosine LR + scheduled-sampling TF decay.
    Early stops on validation loss plateau.
    """
    torch.manual_seed(SEED); np.random.seed(SEED)
    n  = len(combined_train); vs = int(n * (1 - VAL_FRAC))
    t_ds = ClimateDataset(combined_train[:vs],           horizon)
    v_ds = ClimateDataset(combined_train[vs - SEQ_LEN:], horizon)
    tl   = DataLoader(t_ds, BATCH, shuffle=True,  num_workers=0, pin_memory=False)
    vl   = DataLoader(v_ds, BATCH, shuffle=False, num_workers=0, pin_memory=False)

    model = model.to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def _lr_lambda(e):
        if e < WARMUP: return (e + 1) / WARMUP
        p = (e - WARMUP) / max(EPOCHS - WARMUP, 1)
        return max(0.01, 0.5 * (1 + math.cos(math.pi * p)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda)

    best_val = float('inf'); patience_c = 0; best_state = None
    tf_s, tf_e = 0.9, 0.0

    for epoch in range(1, EPOCHS + 1):
        tf = tf_s - (tf_s - tf_e) * (epoch - 1) / max(EPOCHS - 1, 1)

        model.train(); t_loss = 0.
        for xb, yw, yr, fe in tl:
            xb, yw, yr, fe = (t.to(device) for t in (xb, yw, yr, fe))
            opt.zero_grad()
            wo, ro, ra = model(xb, fe, horizon, tf, yw, yr)
            loss = compute_loss(wo, ro, ra, yw, yr)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); t_loss += loss.item()
        t_loss /= len(tl)

        model.eval(); v_loss = 0.
        with torch.no_grad():
            for xb, yw, yr, fe in vl:
                xb, yw, yr, fe = (t.to(device) for t in (xb, yw, yr, fe))
                wo, ro, ra = model(xb, fe, horizon)
                v_loss += compute_loss(wo, ro, ra, yw, yr).item()
        v_loss /= len(vl)
        sched.step()

        if epoch % 10 == 0 or epoch == 1:
            print(f"      [{model_name}] Epoch {epoch:3d}/{EPOCHS} | "
                  f"train={t_loss:.4f}  val={v_loss:.4f}  "
                  f"tf={tf:.2f}  lr={sched.get_last_lr()[0]:.2e}")

        if v_loss < best_val - 1e-5:
            best_val   = v_loss
            best_state = deepcopy(model.state_dict())
            patience_c = 0
        else:
            patience_c += 1
            if patience_c >= PATIENCE:
                print(f"      [{model_name}] Early stop @ epoch {epoch} "
                      f"(best val: {best_val:.4f})")
                break

    model.load_state_dict(best_state)
    os.makedirs(output_dir, exist_ok=True)
    torch.save(model.state_dict(),
               os.path.join(output_dir, f"{model_name.replace('+','')}_best.pt"))
    return model


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 14 — Dynamic Threshold Moving (per-block CSI/F1 grid-search)
# ─────────────────────────────────────────────────────────────────────────────

def _csi(y_true_bin, y_pred_bin):
    tp = int(np.sum(y_true_bin & y_pred_bin))
    fp = int(np.sum(~y_true_bin & y_pred_bin))
    fn = int(np.sum(y_true_bin & ~y_pred_bin))
    denom = tp + fp + fn
    return tp / denom if denom > 0 else 0.0


def tune_thresholds(model: nn.Module, combined_val: np.ndarray,
                    dual_sc: DualScaler, device,
                    val_dates: np.ndarray,
                    roll_horizon: int = ROLL_WINDOW) -> np.ndarray:
    """
    Run the model on the validation split in ROLL_WINDOW chunks, collect all
    predicted sigmoid probabilities and ground-truth rain occurrence, then
    grid-search THRESH_GRID to find the per-block threshold maximising CSI.

    Returns thresholds: (N_RAIN,) float array, one value per rain block.

    val_dates : the REAL calendar dates aligned 1:1 with combined_val's rows
                (BUG 3 FIX — see below).
    """
    model.eval()
    n_val = len(combined_val)
    all_probs = [[] for _ in range(N_RAIN)]
    all_true  = [[] for _ in range(N_RAIN)]

    # Pull original-scale rain from combined_val for ground truth
    # combined_val columns: [scaled_feat (N_FEAT) | exog (N_EXOG)]
    # Inverse-transform the rain columns (cols N_WEATHER:N_FEAT)
    rain_scaled_val = combined_val[:, N_WEATHER:N_FEAT]
    rain_orig_val   = dual_sc.inv_rain(rain_scaled_val)   # original mm

    i = SEQ_LEN
    while i + roll_horizon <= n_val:
        ctx_sc  = combined_val[i - SEQ_LEN: i]            # (SEQ_LEN, N_FEAT+N_EXOG)
        ctx_t   = torch.tensor(ctx_sc, dtype=torch.float32).unsqueeze(0).to(device)

        # ── BUG 3 FIX ────────────────────────────────────────────────────────
        # The original code built exogenous (doy_sin/doy_cos) features from a
        # hardcoded anchor date of 2000-01-01, which does NOT correspond to
        # the real calendar dates of this validation window. For the three
        # models that legitimately consume the exog signal (GRU, LSTM,
        # GRU+LSTM), this fed a seasonal signal shifted away from the true
        # season being validated, corrupting their threshold calibration.
        # Fix: use the REAL validation dates passed in via `val_dates`.
        fut_dates = val_dates[i: i + roll_horizon]
        fe_np = exog_for_dates(fut_dates)                 # (roll_horizon, N_EXOG)
        fe_t = torch.tensor(fe_np, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            _, ro, _ = model(ctx_t, fe_t, roll_horizon)
        probs = ro.squeeze(0).cpu().numpy()          # (roll_horizon, N_RAIN)

        for k in range(N_RAIN):
            all_probs[k].extend(probs[:, k].tolist())
            all_true[k].extend((rain_orig_val[i: i + roll_horizon, k] > 0).tolist())

        i += roll_horizon

    # Grid-search per block
    thresholds = np.full(N_RAIN, 0.5, dtype=np.float32)
    for k in range(N_RAIN):
        probs_k = np.array(all_probs[k])
        true_k  = np.array(all_true[k], dtype=bool)

        # ── BUG 4 FIX ────────────────────────────────────────────────────────
        # The original loop used a strict `>` comparison, so whenever CSI was
        # flat/tied across several thresholds (very likely with an
        # under-calibrated occurrence head), it silently kept the FIRST
        # threshold tried — i.e. the LOWEST value in the ascending
        # THRESH_GRID. That systematically biased every model toward overly
        # permissive gates, which is the most likely explanation for POD and
        # FAR being nearly identical across all six architectures. Fix:
        # collect ALL thresholds tied for the best CSI and take their
        # median, so ties resolve to a representative middle value instead
        # of the grid floor.
        best_csi = -1.0
        tied_thresholds = []
        for t in THRESH_GRID:
            pred_bin = probs_k >= t
            csi = _csi(true_k, pred_bin)
            if csi > best_csi + 1e-9:
                best_csi = csi
                tied_thresholds = [t]
            elif abs(csi - best_csi) <= 1e-9:
                tied_thresholds.append(t)
        thresholds[k] = float(np.median(tied_thresholds)) if tied_thresholds else 0.5

    print(f"      Thresholds (block CSI-optimal, tie-broken via median): "
          f"min={thresholds.min():.2f}  median={np.median(thresholds):.2f}  "
          f"max={thresholds.max():.2f}")
    return thresholds


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 15 — 7-Day Rolling Evaluation  (strict — no autoregressive drift)
# ─────────────────────────────────────────────────────────────────────────────

def rolling_evaluate(model: nn.Module, combined_ctx: np.ndarray,
                      holdout_df: pd.DataFrame, dual_sc: DualScaler,
                      thresholds: np.ndarray, device) -> dict:
    """
    Strict 7-day rolling evaluation:
      For each window of ROLL_WINDOW days:
        1. Context = last SEQ_LEN rows of KNOWN data (no predicted data)
        2. Predict ROLL_WINDOW days forward
        3. Advance by ROLL_WINDOW — next context uses GROUND TRUTH, not pred

    This eliminates compounding autoregressive drift: every window starts
    from verified observations, not from the model's own earlier predictions.

    Returns a dict of metric arrays suitable for the summary table.
    """
    model.eval()
    n_hold  = len(holdout_df)
    n_win   = n_hold // ROLL_WINDOW

    # Collect raw predictions across all windows
    all_w_pred  = np.zeros((n_hold, N_WEATHER))
    all_r_prob  = np.zeros((n_hold, N_RAIN))
    all_r_amt   = np.zeros((n_hold, N_RAIN))

    for w_idx in range(n_win):
        start    = w_idx * ROLL_WINDOW
        end      = start + ROLL_WINDOW
        ctx_rows = combined_ctx[start: start + SEQ_LEN]     # SEQ_LEN known rows
        # Future dates for this window
        fut_dates = holdout_df['Date'].iloc[start:end]
        fe_np  = exog_for_dates(fut_dates.values)            # (ROLL_WINDOW, N_EXOG)

        ctx_t = torch.tensor(ctx_rows, dtype=torch.float32).unsqueeze(0).to(device)
        fe_t  = torch.tensor(fe_np,    dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            wo, ro, ra = model(ctx_t, fe_t, ROLL_WINDOW)

        all_w_pred[start:end]  = wo.squeeze(0).cpu().numpy()
        all_r_prob[start:end]  = ro.squeeze(0).cpu().numpy()
        all_r_amt[start:end]   = ra.squeeze(0).cpu().numpy()

    # Inverse-transform weather
    w_orig = dual_sc.inv_weather(all_w_pred[:n_win * ROLL_WINDOW])
    # Inverse-transform rain amounts
    r_orig = dual_sc.inv_rain(all_r_amt[:n_win * ROLL_WINDOW])
    valid  = slice(0, n_win * ROLL_WINDOW)

    # Apply dynamic threshold gate
    prob_slice = all_r_prob[:n_win * ROLL_WINDOW]
    gate = (prob_slice >= thresholds[None, :]).astype(np.float32)
    r_gated = r_orig * gate                                 # (T, N_RAIN)

    # Ground truth
    gt_w  = holdout_df[WEATHER_FEATURES].values[valid].astype(np.float64)
    gt_r  = holdout_df[RAIN_BLOCKS].values[valid].astype(np.float64)

    # ── Weather metrics ──────────────────────────────────────────────────────
    w_mae  = mean_absolute_error(gt_w, w_orig)
    w_rmse = np.sqrt(mean_squared_error(gt_w, w_orig))

    # ── BUG 5 FIX ────────────────────────────────────────────────────────────
    # The original code applied ONE flat 1.0-unit tolerance to every weather
    # feature regardless of physical unit (°C, %, km/h, W/m² are not
    # comparable on a shared absolute scale) — e.g. 1.0 W/m² is an
    # unreasonably strict bar for Solar Radiation (typical range 0-300+
    # W/m²), which silently deflated the overall Acc% number. Fix: look up
    # each feature's own tolerance from ACC_TOLERANCE.
    tol_vec = np.array([ACC_TOLERANCE[f] for f in WEATHER_FEATURES])
    w_acc   = 100.0 * np.mean(np.abs(gt_w - w_orig) <= tol_vec[None, :])

    # Per-feature MAE for reference
    per_w_mae = {f: mean_absolute_error(gt_w[:, j], w_orig[:, j])
                 for j, f in enumerate(WEATHER_FEATURES)}

    # Per-feature Acc% for reference (BUG 5 FIX — new, mirrors per_w_mae)
    per_w_acc = {f: 100.0 * np.mean(np.abs(gt_w[:, j] - w_orig[:, j]) <= ACC_TOLERANCE[f])
                 for j, f in enumerate(WEATHER_FEATURES)}

    # ── Rainfall occurrence metrics ──────────────────────────────────────────
    true_wet  = gt_r > 0       # (T, N_RAIN) bool
    pred_wet  = gate > 0       # (T, N_RAIN) bool

    pod_list, far_list, csi_list, f1_list = [], [], [], []
    mae_wet_list = []
    for k in range(N_RAIN):
        tw, pw = true_wet[:, k], pred_wet[:, k]
        tp = int(np.sum(tw & pw)); fp = int(np.sum(~tw & pw))
        fn = int(np.sum(tw & ~pw))
        pod = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        far = fp / (tp + fp) if (tp + fp) > 0 else 0.0
        csi = tp / (tp + fn + fp) if (tp + fn + fp) > 0 else np.nan
        prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        f1  = (2 * prec * pod / (prec + pod)
               if prec and pod and not (np.isnan(prec) or np.isnan(pod))
               and (prec + pod) > 0 else np.nan)
        pod_list.append(pod); far_list.append(far)
        csi_list.append(csi); f1_list.append(f1)

        # Wet-day-only MAE
        if tw.sum() >= 2:
            mae_wet_list.append(
                mean_absolute_error(gt_r[tw, k], r_gated[tw, k])
            )
        else:
            mae_wet_list.append(np.nan)

    return {
        # Weather
        "w_mae":     w_mae,
        "w_rmse":    w_rmse,
        "w_acc_1c":  w_acc,        # name kept for backward-compat with existing callers/CSV columns
        "per_w_mae": per_w_mae,
        "per_w_acc": per_w_acc,    # BUG 5 FIX — new per-feature accuracy breakdown
        # Rain occurrence
        "r_pod":     np.nanmean(pod_list),
        "r_far":     np.nanmean(far_list),
        "r_csi":     np.nanmean(csi_list),
        "r_f1":      np.nanmean(f1_list),
        # Wet-day amount
        "r_mae_wet": np.nanmean(mae_wet_list),
        # Raw arrays for further analysis
        "per_pod": pod_list, "per_far": far_list,
        "per_csi": csi_list, "per_f1":  f1_list,
        "per_mae_wet": mae_wet_list,
    }


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 16 — Summary Table Printer
# ─────────────────────────────────────────────────────────────────────────────

def print_summary_table(all_results: dict):
    """
    Prints a comprehensive side-by-side model comparison table.

    Left section : Continuous Weather Variables
                   MAE | RMSE | Acc%(1°C strict tolerance)
    Right section: Rainfall (zero-inflated, per-block averages)
                   POD | FAR | CSI | F1 | Wet-day MAE
    """
    divider = "═" * 110
    print(f"\n{divider}")
    print("  MULTI-MODEL EVALUATION SUMMARY  —  7-day Rolling Window Evaluation")
    print(f"{divider}")
    print(f"  {'Model':<14}  │  "
          f"{'── CONTINUOUS WEATHER ─────────────────────':45}  │  "
          f"{'── RAINFALL (HURDLE DUAL-HEAD) ─────────────────────────':55}")
    print(f"  {'':14}  │  "
          f"{'MAE':>8}  {'RMSE':>8}  {'Acc%(1°C)':>10}  "
          f"{'':12}  │  "
          f"{'POD':>7}  {'FAR':>7}  {'CSI':>7}  {'F1':>7}  {'MAEwet':>8}")
    print(f"  {'─'*14}  │  {'─'*45}  │  {'─'*55}")

    best = {k: (None, None) for k in
            ['w_mae','w_rmse','w_acc_1c','r_pod','r_far','r_csi','r_f1','r_mae_wet']}

    # Find best per metric (lower is better for MAE/RMSE/FAR/MAEwet; higher for rest)
    higher_better = {'w_acc_1c','r_pod','r_csi','r_f1'}
    for k in best:
        vals = [(name, res[k]) for name, res in all_results.items()
                if not np.isnan(res[k])]
        if not vals: continue
        if k in higher_better:
            best[k] = max(vals, key=lambda x: x[1])
        else:
            best[k] = min(vals, key=lambda x: x[1])

    def _fmt(val, key, decimals=3):
        if np.isnan(val): return f"{'N/A':>8}"
        s = f"{val:>{8}.{decimals}f}"
        if best[key] and best[key][0] == model_name:
            s = f"\033[92m{s}\033[0m"   # green highlight for best
        return s

    for model_name, res in all_results.items():
        print(
            f"  {model_name:<14}  │  "
            f"{_fmt(res['w_mae'],    'w_mae')}  "
            f"{_fmt(res['w_rmse'],   'w_rmse')}  "
            f"{_fmt(res['w_acc_1c'], 'w_acc_1c', 1):>10}  "
            f"{'':12}  │  "
            f"{_fmt(res['r_pod'],    'r_pod')}  "
            f"{_fmt(res['r_far'],    'r_far')}  "
            f"{_fmt(res['r_csi'],    'r_csi')}  "
            f"{_fmt(res['r_f1'],     'r_f1')}  "
            f"{_fmt(res['r_mae_wet'],'r_mae_wet'):>8}"
        )

    print(f"  {'─'*14}  │  {'─'*45}  │  {'─'*55}")
    print(f"  Metric target  │  "
          f"{'↓ lower':>8}  {'↓ lower':>8}  {'↑ higher':>10}  "
          f"{'':12}  │  "
          f"{'↑ higher':>7}  {'↓ lower':>7}  {'↑ higher':>7}  {'↑ higher':>7}  {'↓ lower':>8}")
    print(f"{divider}")

    # Per-feature weather breakdown
    print(f"\n  WEATHER MAE per feature:")
    feat_row = f"  {'Model':<14}"
    for f in WEATHER_FEATURES: feat_row += f"  {f[:10]:>11}"
    print(feat_row)
    print(f"  {'─'*14}" + "  " + "  ".join(["─"*11]*N_WEATHER))
    for model_name, res in all_results.items():
        row = f"  {model_name:<14}"
        for f in WEATHER_FEATURES:
            v = res['per_w_mae'].get(f, np.nan)
            row += f"  {v:>11.3f}" if not np.isnan(v) else f"  {'N/A':>11}"
        print(row)

    # BUG 5 FIX: per-feature Acc% breakdown (new — makes the effect of the
    # per-feature tolerance table visible; previously only one flat blended
    # Acc% number existed, hiding which features were being unfairly judged)
    print(f"\n  WEATHER Acc% per feature  (tolerance shown in parentheses):")
    tol_row = f"  {'Model':<14}"
    for f in WEATHER_FEATURES:
        tol_row += f"  {f[:7]+'(±'+str(ACC_TOLERANCE[f])+')':>11}"
    print(tol_row)
    print(f"  {'─'*14}" + "  " + "  ".join(["─"*11]*N_WEATHER))
    for model_name, res in all_results.items():
        row = f"  {model_name:<14}"
        for f in WEATHER_FEATURES:
            v = res.get('per_w_acc', {}).get(f, np.nan)
            row += f"  {v:>10.1f}%" if not np.isnan(v) else f"  {'N/A':>11}"
        print(row)

    # Per-block rainfall breakdown
    print(f"\n  RAINFALL per block  (POD | CSI | MAEwet):")
    print(f"  {'Model':<14}  {'Block':<24}  {'POD':>6}  {'CSI':>6}  {'MAEwet':>8}")
    print(f"  {'─'*14}  {'─'*24}  {'─'*6}  {'─'*6}  {'─'*8}")
    for model_name, res in all_results.items():
        for k, feat in enumerate(RAIN_BLOCKS):
            bl = feat.replace('Rain_', '')
            pod = res['per_pod'][k]; csi = res['per_csi'][k]
            mw  = res['per_mae_wet'][k]
            pod_s = f"{pod:.3f}" if not np.isnan(pod) else "N/A"
            csi_s = f"{csi:.3f}" if not np.isnan(csi) else "N/A"
            mw_s  = f"{mw:.3f}" if not np.isnan(mw)  else "N/A"
            print(f"  {model_name if k == 0 else '':14}  "
                  f"{bl:<24}  {pod_s:>6}  {csi_s:>6}  {mw_s:>8}")
        print()

    print(f"{divider}\n")


# ─────────────────────────────────────────────────────────────────────────────
#  CELL 17 — MAIN  (orchestration loop)
# ─────────────────────────────────────────────────────────────────────────────

def main():
    torch.manual_seed(SEED); np.random.seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("=" * 70)
    print("  BLOCK-LEVEL CLIMATE FORECASTING — MULTI-MODEL EVALUATION")
    print(f"  Device: {device}  |  Models: 6  |  Eval: {ROLL_WINDOW}-day rolling")
    print("=" * 70)

    # ── 1. Load & split ──────────────────────────────────────────────────────
    df = load_data(CSV_PATH)

    holdout_df = df.iloc[-EVAL_HORIZON:].copy()
    train_df   = df.iloc[:-EVAL_HORIZON].copy()

    # Validation split (last VAL_FRAC of training data)
    n_train = len(train_df)
    val_start = int(n_train * (1 - VAL_FRAC))
    val_df    = train_df.iloc[val_start:].copy()
    fit_df    = train_df.iloc[:val_start].copy()

    # ── 2. Scale ─────────────────────────────────────────────────────────────
    print("\n[SCALE] Fitting DualScaler on training data only ...")
    dual_sc = DualScaler()
    sc_train = dual_sc.fit_transform(train_df)    # fit on train (no holdout leakage)
    sc_val   = dual_sc.transform(val_df)
    sc_hold  = dual_sc.transform(holdout_df)      # never used in fit
    sc_full  = dual_sc.transform(df)              # for rolling eval context

    exog_train = train_df[EXOG_COLS].values.astype(np.float32)
    exog_val   = val_df[EXOG_COLS].values.astype(np.float32)
    exog_hold  = holdout_df[EXOG_COLS].values.astype(np.float32)
    exog_full  = df[EXOG_COLS].values.astype(np.float32)

    # BUG 3 FIX: real calendar dates for the validation split, aligned 1:1
    # with comb_val's rows, threaded through to tune_thresholds() instead of
    # the old hardcoded 2000-01-01 anchor date.
    val_dates = val_df['Date'].values

    comb_train = make_combined(sc_train, exog_train)
    comb_val   = make_combined(sc_val,   exog_val)
    comb_full  = make_combined(sc_full,  exog_full)

    print(f"    Train rows: {len(comb_train):,}  "
          f"Val rows: {len(comb_val):,}  "
          f"Holdout rows: {len(sc_hold):,}")

    # ── 3. Context for rolling eval ──────────────────────────────────────────
    # During rolling holdout eval, the context for window w starts at
    # position w*ROLL_WINDOW in the FULL array (so it can look back SEQ_LEN
    # rows into known history before the holdout).
    holdout_offset = len(df) - EVAL_HORIZON
    # Slice the full combined array so that rolling_evaluate() can index
    # combined_ctx[start : start+SEQ_LEN] with start in [0, n_windows-1]*7
    # covering up to SEQ_LEN rows before holdout starts
    eval_ctx_start = holdout_offset - SEQ_LEN
    comb_eval_ctx  = comb_full[eval_ctx_start:]   # (SEQ_LEN + EVAL_HORIZON, N_FEAT+N_EXOG)

    # ── 4. Model registry ────────────────────────────────────────────────────
    MODEL_REGISTRY = [
        GRUModel(),
        LSTMModel(),
        GRULSTMBlend(),
        TCNModel(),
        iTransformerModel(),
        PatchTSTModel(),
    ]

    # ── 5. Main evaluation loop ───────────────────────────────────────────────
    all_results = {}
    for model in MODEL_REGISTRY:
        name = model.name
        print(f"\n{'─'*70}")
        print(f"  [{name}]  Training ...")
        print(f"{'─'*70}")
        t0 = time.time()

        # Train
        model = train_one_model(model, comb_train, ROLL_WINDOW, device,
                                 OUTPUT_DIR, name)

        # Tune per-block thresholds on validation split
        # (BUG 3 FIX: val_dates passed so exog features use REAL dates)
        print(f"  [{name}]  Tuning thresholds on validation split ...")
        thresholds = tune_thresholds(model, comb_val, dual_sc, device,
                                      val_dates, ROLL_WINDOW)

        # 7-day rolling holdout evaluation
        print(f"  [{name}]  Evaluating on {EVAL_HORIZON}-day holdout "
              f"({EVAL_HORIZON // ROLL_WINDOW} windows × {ROLL_WINDOW} days) ...")
        results = rolling_evaluate(model, comb_eval_ctx, holdout_df,
                                    dual_sc, thresholds, device)
        all_results[name] = results
        elapsed = time.time() - t0

        print(f"  [{name}]  Done in {elapsed:.1f}s  |  "
              f"Weather MAE={results['w_mae']:.3f}  "
              f"Rain POD={results['r_pod']:.3f}  "
              f"CSI={results['r_csi']:.3f}  "
              f"MAEwet={results['r_mae_wet']:.3f}")

        # Free GPU memory between models
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ── 6. Summary table ─────────────────────────────────────────────────────
    print_summary_table(all_results)

    # ── 7. Save results CSV ──────────────────────────────────────────────────
    summary_rows = []
    for name, res in all_results.items():
        row = {"Model": name,
               "W_MAE": round(res['w_mae'], 4),
               "W_RMSE": round(res['w_rmse'], 4),
               "W_Acc_1C": round(res['w_acc_1c'], 2),
               "R_POD": round(res['r_pod'], 4),
               "R_FAR": round(res['r_far'], 4),
               "R_CSI": round(res['r_csi'], 4),
               "R_F1": round(res['r_f1'], 4),
               "R_MAEwet": round(res['r_mae_wet'], 4)}
        for f in WEATHER_FEATURES:
            row[f"MAE_{f}"] = round(res['per_w_mae'].get(f, np.nan), 4)
        for k, feat in enumerate(RAIN_BLOCKS):
            b = feat.replace('Rain_','')
            row[f"POD_{b}"] = round(res['per_pod'][k], 4) if not np.isnan(res['per_pod'][k]) else np.nan
            row[f"CSI_{b}"] = round(res['per_csi'][k], 4) if not np.isnan(res['per_csi'][k]) else np.nan
        summary_rows.append(row)

    out_csv = os.path.join(OUTPUT_DIR, "model_comparison.csv")
    pd.DataFrame(summary_rows).to_csv(out_csv, index=False)
    print(f"  Results saved → {out_csv}")
    print(f"  Model weights → {OUTPUT_DIR}/<model>_best.pt\n")


if __name__ == "__main__":
    main()

  BLOCK-LEVEL CLIMATE FORECASTING — MULTI-MODEL EVALUATION
  Device: cpu  |  Models: 6  |  Eval: 7-day rolling

[DATA] Loading: master_weather_blocks.csv
    Rows=5,114  Range=2012-01-01 → 2025-12-31
    Avg zero-rain: 58.3%

[SCALE] Fitting DualScaler on training data only ...
    Train rows: 4,994  Val rows: 750  Holdout rows: 120

──────────────────────────────────────────────────────────────────────
  [GRU]  Training ...
──────────────────────────────────────────────────────────────────────
      [GRU] Epoch   1/80 | train=1.2892  val=1.0889  tf=0.90  lr=2.00e-04
